# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [1]:
import os
import sys
import pandas as pd

from model.Model import model_MSG
#from model.Plotter import plotter_MSG
#from model.coders_linkage import CODERS_pull
from model.policies.OBPS import apply_obps
from message_ix import make_df

# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all

c:\Users\crist\anaconda3\envs\message_ix\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


<IPython.core.display.Javascript object>

In [2]:
# Sys and OS Operations
os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [39]:
# Set-up
msg = model_MSG(scenario_name='dacs', config=False)
platform, msg_scenario, msg_folder = msg.setup_scenario()



Model Logs: Setting up scenario

Model Logs: Adding technology


		Model creation time: 6 Min and 9 Sec 		
Model Logs: Solving the model...


				Solving process time: 27 Min and 23 Sec				


## Relations

After a model is solved, if there exist any relations for newly added technologies (added via the scenario configuration), then these need to be added in the next step

In [42]:
# Add relations

DAC_mpen= {
"relation_name" : "DAC_mpen_c",
"growth_act_up" : 0.05,
"initial_act" : 0.12033,
"relation_lower": 0,

# Relation activities by technology
"technologies" : ["DAC_mpen", "dac_ht", "dac_lt"], 
"DAC_mpen_value" : -1,
"dac_ht_value" : 1,
"dac_lt_value" : 1,
}

provinces = provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5))

yearly_relations = []

for year in years:

    dac_ht_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['dac_ht'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_ht_df)

    dac_lt_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['dac_lt'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_lt_df)
    
    dac_mpen_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['DAC_mpen'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[-1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_mpen_df)

# Concatenate all DataFrames
combined_relation_activities = pd.concat(yearly_relations, ignore_index=True)



In [15]:
# Data viewer
combined_relation_activities
#yearly_relations[0]

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
0,DAC_mpen_c,Ontario,2020,Ontario,dac_ht,2020,M1,1,-
1,DAC_mpen_c,Quebec,2020,Quebec,dac_ht,2020,M1,1,-
2,DAC_mpen_c,BritishColumbia,2020,BritishColumbia,dac_ht,2020,M1,1,-
3,DAC_mpen_c,Alberta,2020,Alberta,dac_ht,2020,M1,1,-
4,DAC_mpen_c,Manitoba,2020,Manitoba,dac_ht,2020,M1,1,-
...,...,...,...,...,...,...,...,...,...
346,DAC_mpen_c,NewfoundlandandLabrador,2060,NewfoundlandandLabrador,DAC_mpen,2060,M1,-1,-
347,DAC_mpen_c,PrinceEdwardIsland,2060,PrinceEdwardIsland,DAC_mpen,2060,M1,-1,-
348,DAC_mpen_c,NorthwestTerritories,2060,NorthwestTerritories,DAC_mpen,2060,M1,-1,-
349,DAC_mpen_c,Yukon,2060,Yukon,DAC_mpen,2060,M1,-1,-


In [45]:
#msg_scenario.remove_solution()
#msg_scenario.check_out()

# Add the combined DataFrame to the scenario
msg_scenario.add_par('relation_activity', combined_relation_activities)

# Loop through provinces and years to add the activity_up parameters
for province in provinces:
    for year in years:
        for technology in DAC_mpen["technologies"]:
            
            msg_scenario.add_par('growth_activity_up', 
                        [province, technology, year, "year"], 
                        DAC_mpen["growth_act_up"], 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            
            msg_scenario.add_par('initial_activity_up', 
                    [province, technology, year, "year"], 
                    DAC_mpen["initial_act"], 'GWa')
        

ValueError: The index set 'relation' does not have an element 'DAC_mpen_c'!

In [10]:
# Solve model with added relations
#msg_scenario.check_out()
msg_scenario.solve(solve_options={"write_model_to_file":"true", "writelp":"true", "write_model_file":"true", "lpmethod":2}, case="DACS_with_mpen")

In [ ]:
#msg_scenario.cat_list(name="emission")
msg_scenario.

['MERtoPPP',
 'abs_cost_activity_soft_lo',
 'abs_cost_activity_soft_up',
 'abs_cost_new_capacity_soft_lo',
 'abs_cost_new_capacity_soft_up',
 'addon_conversion',
 'addon_lo',
 'addon_up',
 'aeei',
 'bound_activity_lo',
 'bound_activity_up',
 'bound_emission',
 'bound_extraction_up',
 'bound_new_capacity_lo',
 'bound_new_capacity_up',
 'bound_total_capacity_lo',
 'bound_total_capacity_up',
 'capacity_factor',
 'commodity_stock',
 'construction_time',
 'cost_MESSAGE',
 'demand',
 'depr',
 'drate',
 'duration_period',
 'duration_time',
 'dynamic_land_lo',
 'dynamic_land_up',
 'emission_factor',
 'emission_scaling',
 'esub',
 'fix_cost',
 'fixed_activity',
 'fixed_capacity',
 'fixed_extraction',
 'fixed_land',
 'fixed_new_capacity',
 'fixed_stock',
 'flexibility_factor',
 'gdp_calibrate',
 'grow',
 'growth_activity_lo',
 'growth_activity_up',
 'growth_land_lo',
 'growth_land_scen_lo',
 'growth_land_scen_up',
 'growth_land_up',
 'growth_new_capacity_lo',
 'growth_new_capacity_up',
 'histori

In [ ]:
#msg_scenario.remove_solution()
#msg_scenario.check_out()

# Adding emissions for DAC storage

co2_stor_emission_fact_list = [] 

for m in ["M1", "M2", "M3"]:
    temp = make_df(
        "emission_factor",
        technology="co2_stor",
        node_loc=provinces,
        emission=["CO2_storage"] * len(provinces),
        value=[1] * len(provinces),
        unit=["-"] * len(provinces),
        mode= m,
        year_vtg=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    co2_stor_emission_fact_list.append(temp)

co2_stor_emission_fact = pd.concat(co2_stor_emission_fact_list, ignore_index=True)

msg_scenario.add_set('emission', 'CO2_storage')
msg_scenario.add_set('type_emission', 'co2_storage_pot')
msg_scenario.add_set('type_tec', 'co2_potential')

emis_dict = {"type_emission": 'co2_storage_pot', 'emission': 'CO2_storage'}
tec_dict = {"type_tec": 'co2_potential', 'technology': 'co2_stor'}

msg_scenario.add_par('cat_emission', pd.DataFrame({k:[v] for k,v in emis_dict.items()}))
msg_scenario.add_par('cat_tec', pd.DataFrame({k:[v] for k,v in tec_dict.items()}))

for prov in provinces:
    for yr in years:
        msg_scenario.add_par('bound_emission', [prov, "co2_storage_pot", "co2_potential", yr], 163.84, '-')

msg_scenario.add_par('emission_factor', co2_stor_emission_fact)




TypeError: Scenario.add_cat() missing 1 required positional argument: 'keys'

In [20]:
msg_scenario.solve(solve_options={"write_model_to_file":"true", "writelp":"true", "write_model_file":"true", "lpmethod":2}, case="DACS_with_mpen_and_storage_emission")